# Question 3: when in a vampire's lifetime does the biting happen?

MATH 1660, November 2026.

A vampire can bite for five days (120 hours) after turning. Did the bites spread out over those five days, or did they come all at once, near the turn? This notebook answers that from our research export, with one test, a check of the test by simulation, and one figure.

**How this connects to *disease or fad?*** The SEIR model we built in week 2 has each vampire bite at a steady rate for the whole of $T_I$. This notebook tests that assumption. It does not decide disease against fad: that question, as first asked, is about the person joining (one exposure or several), and our data have no declines to answer it. And the result is about our model's assumption, not about diseases in general, whose infectiousness also changes over time.

**Use practice mode to write your plan.** The data from our game are in the **research export**: a set of CSV files made by the game's platform, recording who played, who bit whom and when, and when each player turned and burned out. Each team is given a web link from which this notebook can read those files, but only after the team's plan is submitted.

Until then, the notebook does not read our game's data. While `PRACTICE = True` in Step 1, it makes up an export for an imaginary game, with the same files and the same columns, and runs every step on that. Use this to learn what each step computes: run the notebook, change the practice settings in Step 1, and run it again. Then decide the choices in Step 1 that your plan will state, such as `WINDOW_HOURS`.

After your plan is submitted, you are given the link. Paste it into Step 1, set `PRACTICE = False`, and run the notebook again: it now reads our game's export, and uses the choices your plan states.

**The result from the choices in your plan is your test.** Suppose that after seeing it you run the notebook again with a different choice, for example a different `WINDOW_HOURS` in Step 1. The result of that second run is **exploratory**, and your report says so. It gives the result from your plan's choices first. Then it gives the second result, labeled *exploratory*, with the choice you changed and why. An exploratory result is not a test. Its p-value is not the chance that Step 5 describes, because the choice was made after seeing the data, and a choice made that way can be the one that happens to give the smallest p-value. An exploratory result can still suggest a question worth testing in another game.

Nobody has checked this notebook except its author. If a number looks wrong, it may be.

## How to use this notebook

1. **Make your own copy first:** *File*, then *Save a copy in Drive*.
2. Run the cells in order, from the top: click inside a cell and press **Shift+Enter**.
3. Change only the values in Step 1.

## Step 1: your choices, and the export

- `PRACTICE`: while it is `True`, the notebook makes up an export for an imaginary game and does not read our game's export, so you can learn every step without seeing our game's result. In the made-up export, every bite falls at a random time in its vampire's lifetime: the null hypothesis of Step 5 is true by construction. `PRACTICE_SEED` picks which made-up export you get; change it to get another. Set `PRACTICE = False` only after your team's plan is submitted.
- `EXPORT_LINK`: empty until your plan is submitted. Then paste your team's export link between the quotes. It ends with `/`. **Do not share a notebook that has the link in it**: anyone with the link can read the research data. If you would rather upload the files, leave the link empty and upload `settings.csv`, `participants.csv`, `transmissions.csv`, `state_changes.csv`, and `withdrawals.csv` with the folder icon on the left.
- `WINDOW_HOURS`: the length of the window, written $w$ below. The window is the first $w$ hours after a vampire turns: from hour 0, the moment of turning, up to hour $w$. A bite at hour $w$ exactly is outside it. Use the value in your plan.
- `REPS`: how many times the simulation in Step 6 repeats.

In [ ]:
# Step 1: your choices
PRACTICE = True         # True: a made-up export. False: our export (only after your plan is handed in)
PRACTICE_SEED = 11      # change it to get a different made-up export
EXPORT_LINK = ""        # after your plan is submitted: paste your team's export link here, or upload the files
WINDOW_HOURS = 1        # the value in your plan
REPS = 10000            # repetitions of the simulation in Step 6

In [ ]:
# Step 1b: read the export, or make one up
import csv, io, math, random, urllib.request
from datetime import datetime, timedelta, timezone

FILES = ["settings.csv", "participants.csv", "transmissions.csv", "state_changes.csv", "withdrawals.csv"]

def read(name):
    if EXPORT_LINK:
        text = urllib.request.urlopen(EXPORT_LINK + name).read().decode("utf-8")
    else:
        text = open(name, encoding="utf-8").read()
    return list(csv.DictReader(io.StringIO(text)))

def made_up_export(seed):
    """30 vampires, each with a 120-hour lifetime that ends before the spree. 20 turned the
    moment they joined; 10 were bitten and turned 12 hours later. Most bite no one. Every bite
    falls at a random time in its vampire's lifetime, so the null hypothesis of Step 5 is true."""
    rng = random.Random(seed)
    iso = lambda d: d.isoformat()
    launch = datetime(2026, 10, 16, 23, 0, tzinfo=timezone.utc)
    out = {"settings.csv": [{"spree_start_at": "2026-11-09T18:03:00+00:00",
                             "end_at": "2026-11-14T00:00:00+00:00",
                             "refreshed_at": "2026-11-11T12:00:00+00:00"}],
           "participants.csv": [], "transmissions.csv": [], "state_changes.csv": [], "withdrawals.csv": []}
    for i in range(30):
        pid = f"practice-{i + 1:02d}"
        in_person = i < 20
        turn = launch + timedelta(hours=rng.uniform(0, 17 * 24))
        joined = turn if in_person else turn - timedelta(hours=12)
        out["participants.csv"].append({"participant_id": pid, "is_seed": str(in_person),
                                        "joined_through": "walker" if in_person else "",
                                        "joined_at": iso(joined), "turn_at": iso(turn)})
        out["state_changes.csv"].append({"participant_id": pid, "change": "vampire → burned out",
                                         "at": iso(turn + timedelta(hours=120))})
        for _ in range(rng.choice([0] * 7 + [1, 1, 2, 3])):
            out["transmissions.csv"].append({"source_id": pid, "target_id": "", "target_already_in_game": "False",
                                             "at": iso(turn + timedelta(hours=rng.uniform(0, 120)))})
    return out

data = made_up_export(PRACTICE_SEED) if PRACTICE else {name: read(name) for name in FILES}
t = lambda s: datetime.fromisoformat(s) if s else None
hours = lambda a, b: (b - a).total_seconds() / 3600     # hours from a to b

settings = data["settings.csv"][0]
players = {p["participant_id"]: p for p in data["participants.csv"]}
bites = data["transmissions.csv"]
if PRACTICE:
    print(f"PRACTICE MODE: a made-up export (seed {PRACTICE_SEED}). None of these numbers comes from our game.")
else:
    print("Our export, made at", settings["refreshed_at"], "(UTC)")
print(len(players), "players,", len(bites), "bites in all")

## Step 2: which vampires are finished

A vampire is **finished** when their five days are over, so their number of bites can no longer change. The spree, which started at `spree_start_at` in `settings.csv`, let burned-out players bite again, so a vampire counts as finished here only if they burned out *before the spree started*.

Use `state_changes.csv` for this, not the `burnout_at` column of `participants.csv`. For players the spree brought back, `burnout_at` now shows the end of the game, not the time they first burned out.

A **bite** is a row of `transmissions.csv` that made its target a new player (`target_already_in_game` is `False`). A row with `True` is a scan of someone already in the game, and is not a bite. A finished vampire's bites are the bites they made **before their first burnout**. A bite they made during the spree, after the spree brought them back, is not part of their lifetime and is left out. A **biter** is a finished vampire with at least one such bite.

In [ ]:
# Step 2: finished vampires
spree = t(settings["spree_start_at"])
first_burnout = {}
for row in data["state_changes.csv"]:
    if row["change"] == "vampire → burned out":
        first_burnout.setdefault(row["participant_id"], t(row["at"]))

finished = [p for p in players.values()
            if p["participant_id"] in first_burnout and first_burnout[p["participant_id"]] < spree]
bites_by = {}                        # each vampire's bites before their first burnout
for b in bites:
    src = b["source_id"]
    if b["target_already_in_game"] == "False" and src in first_burnout and t(b["at"]) < first_burnout[src]:
        bites_by.setdefault(src, []).append(t(b["at"]))

biters = [p for p in finished if p["participant_id"] in bites_by]
print(len(finished), "finished vampires;", len(biters), "of them bit someone;",
      sum(len(bites_by[p["participant_id"]]) for p in biters), "bites between them")

## Step 3: when each biter bit

For each biter: how long after joining they turned, how long their lifetime was, and the time of each bite in hours after turning. A vampire who turned at the moment they joined was recruited in person, and the person who recruited them was usually with them.

### The statistic $f$

Number the biters $1, 2, \dots, n$. For biter $i$:

- $b_i$ is the number of bites biter $i$ made;
- $L_i$ is the length of biter $i$'s lifetime, in hours, from turning to their first burnout;
- $t_{i,1}$ is the time of biter $i$'s earliest bite, in hours after biter $i$ turned;
- $X_i = 1$ if $t_{i,1} < w$, and $X_i = 0$ otherwise.

So $X_i = 1$ when biter $i$ bit someone less than $w$ hours after turning, and $X_i = 0$ when every bite by biter $i$ came at hour $w$ or later. (The earliest bite is less than $w$ hours after turning exactly when at least one bite is, so $X_i = 1$ also means "biter $i$ bit at least once in the window".) The statistic is

$$f = X_1 + X_2 + \dots + X_n ,$$

the number of biters whose earliest bite came less than $w$ hours after they turned. It is a whole number from 0 to $n$.

*Example,* with $w = 1$ and four made-up biters:

| Biter $i$ | Bite times, hours after turning | $t_{i,1}$ | $X_i$ |
|---|---|---|---|
| 1 | 0.05, 0.10 | 0.05 | 1 |
| 2 | 0.40 | 0.40 | 1 |
| 3 | 2.0, 50.0, 51.0 | 2.0 | 0 |
| 4 | 88.0 | 88.0 | 0 |

Here $f = 1 + 1 + 0 + 0 = 2$. Biter 1 counts once, not twice, although both of their bites are in the window. Biter 3's bites at hours 50 and 51 do not change $X_3$. Hour 2 and hour 88 both give $X_i = 0$.

The cell below prints the same columns for our biters and adds them up. In the code, `r["bites, hours after turning"]` is biter $i$'s bite times sorted from earliest to latest, so `[0]` is $t_{i,1}$, and `< WINDOW_HOURS` is True (counted as 1) or False (counted as 0): that is $X_i$.

In [ ]:
# Step 3: the bite times, in hours after turning
rows = []
for p in biters:
    turn = t(p["turn_at"])
    rows.append({
        "turned, hours after joining": hours(t(p["joined_at"]), turn),
        "lifetime, hours": hours(turn, first_burnout[p["participant_id"]]),
        "bites, hours after turning": sorted(hours(turn, b) for b in bites_by[p["participant_id"]]),
    })
rows.sort(key=lambda r: r["bites, hours after turning"][0])

X = [int(r["bites, hours after turning"][0] < WINDOW_HOURS) for r in rows]   # X_i for each biter
f = sum(X)

print(f"{'biter i':>7}  {'turned after joining':>20}  {'L_i':>6}  {'t_i,1':>6}  {'X_i':>3}  bites, hours after turning")
for i, (r, x) in enumerate(zip(rows, X), 1):
    times = ", ".join(f"{h:.2f}" for h in r["bites, hours after turning"])
    print(f"{i:>7}  {r['turned, hours after joining']:>18.1f} h  {r['lifetime, hours']:>4.0f} h  "
          f"{r['bites, hours after turning'][0]:>6.2f}  {x:>3}  {times}")

at_join = sum(r["turned, hours after joining"] < 1 / 60 for r in rows)
print()
print(f"f = {' + '.join(map(str, X))} = {f}")
print(f"{f} of the {len(rows)} biters made their earliest bite less than {WINDOW_HOURS} hour(s) after turning.")
print(f"{at_join} of {len(rows)} biters turned at the moment they joined.")

## Step 4: the figure

One row for each biter, one dot for each bite, at the hour after turning when it happened. The dashed line is the end of the window from Step 1. If the bites were spread over the lifetime, the dots would be spread across the whole width.

In [ ]:
# Step 4: the figure
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 0.5 * len(rows) + 1.5))
for i, r in enumerate(rows, 1):
    ax.plot(r["bites, hours after turning"], [i] * len(r["bites, hours after turning"]), "o", color="black")
ax.axvline(WINDOW_HOURS, linestyle="--", color="gray")
ax.set_xlim(-3, 123)
ax.set_xlabel("hours after turning")
ax.set_ylabel("biter")
ax.set_yticks(range(1, len(rows) + 1))
ax.set_title("When each finished vampire bit")
plt.show()

## Step 5: the test, and how its p-value is computed

### 5a. What the test asks

**The null hypothesis:** each vampire's bites fall at random times in their lifetime, every hour equally likely, whatever happened before. If that were so, a bite in the first hour after turning would be no more likely than a bite in any other hour.

**The unit is the biter, not the bite.** Bites made a few minutes apart by the same vampire, in the same place, are not separate random events, so counting each bite would overstate how much data we have. So each biter $i$ contributes one number, $X_i$, which is 1 or 0 (Step 3), whatever their number of bites.

**The statistic** is $f = X_1 + X_2 + \dots + X_n$, the number of biters whose earliest bite came less than $w$ hours after they turned (Step 3).

Write $f_{\text{obs}}$ for the observed value of $f$: the value Step 3 computed from the export this notebook read (in practice mode, the made-up export).

**The p-value answers one question:** *if the null hypothesis were true, what is the chance that $f$ would be equal to $f_{\text{obs}}$ or larger?* In words: the chance that $f_{\text{obs}}$ biters or more would make their earliest bite less than $w$ hours after turning. If that chance is very small, the observed data would be rare under the null hypothesis, and we reject it. We count "$f_{\text{obs}}$ or more", not "exactly $f_{\text{obs}}$", because any value of $f$ at least as large as $f_{\text{obs}}$ would be at least as strong a sign of biting at the turn. We do not count smaller numbers, because they would point the other way.

The p-value takes some things as given: which vampires bit, how many times each bit, and how long each lifetime was. It asks only about *when* the bites happened. That is what *conditional on* those facts means.

### 5b. The chance that $X_i = 1$, for one biter

Take biter $i$, with $b_i$ bites and a lifetime of $L_i$ hours, and a window of $w$ hours. Under the null hypothesis, each of the $b_i$ bites lands at a random time in the $L_i$ hours, every time equally likely.

- The chance that one bite lands **outside** the window, at hour $w$ or later, is $1 - w/L_i$.
- The bites land independently, so the chance that **all** $b_i$ land outside is $(1 - w/L_i)^{b_i}$. That is the chance that $X_i = 0$.
- $X_i = 1$ in every other case, so the chance that $X_i = 1$ is

$$c_i = \Pr(X_i = 1) = 1 - \left(1 - \frac{w}{L_i}\right)^{b_i} .$$

*Example:* $w = 1$, $L_i = 120$, $b_i = 3$. One bite lands outside with chance $119/120 = 0.99167$; all three with chance $0.99167^3 = 0.97521$; so $c_i = 1 - 0.97521 = 0.0248$, about 2.5 percent. With $b_i = 1$, $c_i = 1/120 = 0.0083$.

The cell below computes $c_i$ for each biter.

In [ ]:
# Step 5b: c_i, the chance that X_i = 1 for biter i, under the null hypothesis
chances = []
for i, r in enumerate(rows, 1):
    b, L = len(r["bites, hours after turning"]), r["lifetime, hours"]
    c = 1 - (1 - WINDOW_HOURS / L) ** b
    chances.append(c)
    print(f"biter {i}: b_i = {b}, L_i = {L:.0f} hours, c_i = 1 - (1 - {WINDOW_HOURS}/{L:.0f})^{b} = {c:.4f}")

### 5c. From one biter to all of them

$f = X_1 + X_2 + \dots + X_n$, and each $X_i$ is 1 with chance $c_i$ and 0 with chance $1 - c_i$, independently of the others. The chances $c_i$ differ from biter to biter, so we cannot use a single formula such as the binomial; we build the chances of each value of $f$ up one biter at a time.

*Two biters,* with chances $c_1$ and $c_2$. There are four possible outcomes:

$$
\begin{array}{cc|c|l}
X_1 & X_2 & f = X_1 + X_2 & \text{Chance} \\
\hline
0 & 0 & 0 & (1 - c_1)(1 - c_2) \\
1 & 0 & 1 & c_1 (1 - c_2) \\
0 & 1 & 1 & (1 - c_1)\, c_2 \\
1 & 1 & 2 & c_1 c_2
\end{array}
$$

With $c_1 = 0.0248$ and $c_2 = 0.0166$: $\Pr(f = 0) = 0.9752 \times 0.9834 = 0.95901$; $\Pr(f = 2) = 0.0248 \times 0.0166 = 0.00041$; and $\Pr(f = 1)$ is the rest, $0.04058$. The three add up to 1.

*Adding a third biter,* with chance $c_3$. To end with $X_1 + X_2 + X_3 = j$, either $X_1 + X_2 = j$ and $X_3 = 0$, or $X_1 + X_2 = j - 1$ and $X_3 = 1$:

$$\Pr(X_1 + X_2 + X_3 = j) = \Pr(X_1 + X_2 = j)\,(1 - c_3) + \Pr(X_1 + X_2 = j - 1)\,c_3 .$$

The same rule adds the fourth biter, then the fifth, and so on. With six biters there are $2^6 = 64$ outcomes; the rule adds up all of them without listing them.

*The p-value* is the chance, under the null hypothesis, that $f$ is $f_{\text{obs}}$ or more: $p = \Pr(f = f_{\text{obs}}) + \Pr(f = f_{\text{obs}} + 1) + \dots + \Pr(f = n)$.

The cell below does exactly this and prints the whole table. The row marked with an arrow is $f_{\text{obs}}$; the last column on that row is the p-value.

In [ ]:
# Step 5c: the chance of each count, built up one biter at a time, and the p-value
dist = [1.0]                         # dist[j] = chance that the sum so far is j; before any biter, 0 for certain
for c in chances:
    new = [0.0] * (len(dist) + 1)
    for j, pj in enumerate(dist):
        new[j] += pj * (1 - c)       # this biter has X_i = 0: the sum stays j
        new[j + 1] += pj * c         # this biter has X_i = 1: the sum goes up to j + 1
    dist = new

print(f"{'f':>5}  {'chance f is exactly this':>26}  {'chance f is this or more':>26}")
for j in range(len(dist)):
    mark = "   <- observed f" if j == f else ""
    print(f"{j:>5}  {dist[j]:>26.2e}  {sum(dist[j:]):>26.2e}{mark}")

expected = sum(chances)              # the average of f under the null hypothesis: c_1 + c_2 + ... + c_n
p = sum(dist[f:])
print()
print(f"Average of f if bites are at random times: {expected:.2f}. Observed f = {f}.")
print(f"p-value, the chance of {f} or more: " + (f"{p:.4f}" if p >= 0.001 else f"{p:.2e}"))

### 5d. Reading the p-value

- **Below 0.05** (or the level in your plan): a value of $f$ as large as $f_{\text{obs}}$ would be rare if bites fell at random times, so we reject the null hypothesis. Biting was concentrated in the window.
- **At or above it:** a value of $f$ as large as $f_{\text{obs}}$ is not rare under the null hypothesis, so we do not reject it. That does not show that bites fell at random times; with few biters, the test may not be able to tell.
- **If $f = 0$**, the p-value is 1: every possible count is "0 or more".

In practice mode, the null hypothesis is true by construction, so the p-value should usually be large. Change `PRACTICE_SEED` in Step 1 and run everything again a few times: about how often is the p-value below 0.05?

## Step 6: check the p-value by simulation

Step 5 computed the p-value from formulas. This step gets it a second way, by doing what the null hypothesis says, many times, and counting:

1. For each biter $i$, place their $b_i$ bites at random times in their $L_i$ hours.
2. For each biter, set $X_i = 1$ if their earliest simulated bite is less than $w$ hours after turning, and $X_i = 0$ otherwise. The sum $X_1 + \dots + X_n$ is one simulated $f$.
3. Repeat `REPS` times.
4. The share of repetitions in which the simulated $f$ is equal to $f_{\text{obs}}$ or larger is the simulated p-value.

It should be close to the p-value of Step 5, and the average simulated $f$ should be close to the average printed in Step 5c. If they disagree by much, something in Step 5 is wrong. When the p-value is very small, the simulation may find no repetition with $f$ or more; that shows only that $p$ is below $1/\text{REPS}$, which the exact value in Step 5 makes precise.

In [ ]:
# Step 6: simulation of the null hypothesis
rng = random.Random(1)
counts = []
for _ in range(REPS):
    f_sim = 0
    for r in rows:
        times = [rng.uniform(0, r["lifetime, hours"]) for _ in r["bites, hours after turning"]]
        f_sim += min(times) < WINDOW_HOURS     # X_i for this biter: 1 if the earliest bite is in the window
    counts.append(f_sim)
share = sum(c >= f for c in counts) / REPS
print(f"In {REPS} repetitions, the average simulated f was {sum(counts) / REPS:.2f} (Step 5c: {expected:.2f}).")
print(f"Simulated p-value, the share of repetitions with {f} or more: {share}"
      + (f"  (below 1/{REPS})" if share == 0 else f"   (Step 5: {p:.4f})"))

## Step 7: what the test does not say

A small p-value says the bites were not spread at random over the lifetime. It does not say **why**. Step 3 shows how many biters turned at the moment they joined: for them, the moment of turning was also the moment the person who recruited them was standing with them, and perhaps other people too. So "biting happens at the turn" and "biting happens when the vampire is with other people" cannot be told apart in these data. A report says so.

Two more things a report should state:

- **How many biters the test rests on.** It is the number of rows in Step 3, not the number of bites.
- **The window.** The result with the `WINDOW_HOURS` in your plan is the test. A result with any other value, run after seeing the first, is exploratory (see the top of the notebook): the report gives it second, labeled *exploratory*, with the value used and why.

## Step 8: your result

Fill this in, in your own words, in a text cell below:

- The number of finished vampires, how many bit, and how many bites.
- The window $w$, and $f$: the number of biters whose earliest bite came less than $w$ hours after they turned.
- The average of $f$ if bites were at random times, and the p-value (Step 5), and whether the simulation (Step 6) agrees.
- What your plan said each result would mean, and which one happened.
- What the result does not say (Step 7).

## Step 9 (optional): the spree

*Optional; no part of a report depends on it.* Only for a team that wrote a spree prediction before its period started, and only on an export made after the period ended.

From `spree_start_at`, every player could bite again, with no wait. If biting happens mostly at the turn, players who turned long ago should bite during the spree at about the rate vampires bit *after* the window before the spree. If biting happens at any time, they should bite at about the rate over whole lifetimes. Set `PREDICTION_START` to the time your team wrote its prediction, in UTC (Central time plus 6 hours), and `PREDICTION_END` to the end of the period. The cell computes both rates from the game before the spree, the vampire-days in your period, the number of bites each rate expects, and the chance of a count as low as, or as high as, the one observed, if bites come at that rate independently (a Poisson count).

In [ ]:
# Step 9 (optional): the spree
PREDICTION_START = "2026-11-11T18:00:00+00:00"   # when your team wrote its prediction, in UTC
PREDICTION_END = settings["end_at"]               # the end of your period; the game ends at end_at

start, end = t(PREDICTION_START), t(PREDICTION_END)
if t(settings["refreshed_at"]) < end:
    print("This export was made before the end of the period. Run this step on a later export.")
else:
    withdrawn = {w["participant_id"]: t(w["at"]) for w in data["withdrawals.csv"]}

    def days(a, b):
        return max(0.0, hours(a, b)) / 24

    # before the spree: every vampire from the turn to the first burnout, the spree start, or withdrawal
    vd_all = vd_late = 0.0
    for p in players.values():
        turn = t(p["turn_at"])
        if turn is None or turn >= spree:
            continue
        stop = min(x for x in (first_burnout.get(p["participant_id"]), spree,
                               withdrawn.get(p["participant_id"])) if x)
        vd_all += days(turn, stop)
        vd_late += max(0.0, hours(turn, stop) - WINDOW_HOURS) / 24
    before = [b for b in bites if t(b["at"]) < spree]
    late = [b for b in before if hours(t(players[b["source_id"]]["turn_at"]), t(b["at"])) >= WINDOW_HOURS]
    rate_late, rate_all = len(late) / vd_late, len(before) / vd_all

    # the period: every player in the game, from when they joined (or the period start) to its end
    vd_period = sum(days(max(start, t(p["joined_at"])), min(end, withdrawn.get(p["participant_id"], end)))
                    for p in players.values())
    observed = sum(start <= t(b["at"]) < end for b in bites)

    def poisson_cdf(x, mean):
        return sum(math.exp(-mean) * mean ** i / math.factorial(i) for i in range(x + 1))

    print(f"Vampire-days in the period: {vd_period:.1f}. Bites observed in it: {observed}.")
    for name, rate in (("after the window", rate_late), ("over whole lifetimes", rate_all)):
        mean = rate * vd_period
        low = poisson_cdf(observed, mean)
        high = 1 - poisson_cdf(observed - 1, mean) if observed > 0 else 1.0
        print(f"At the rate {name} before the spree ({rate:.4f} bites per vampire-day): "
              f"expected {mean:.2f}; chance of {observed} or fewer {low:.3f}, of {observed} or more {high:.3f}")